# Random Event GPPO Preliminary Training

This notebook runs the preliminary three-seed experiment for GPPO-NoGate, GPPO-Adaptive, and Fair PPO-MLP.

**Important**: This is a `preliminary` run with only 3 training seeds. Results should not be used as final conclusions.

In [ ]:
# Install dependencies
!pip install -q torch numpy gymnasium stable-baselines3 sb3-contrib

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Mount Google Drive for checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/random_event_gppo_preliminary'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_PATH}')

In [ ]:
# Clone or upload project files
# Option 1: Clone from repository
# !git clone <repository_url> /content/54_20-master

# Option 2: Upload files manually
# Use the file upload widget to upload the project files

# For this notebook, we assume the project is already available
PROJECT_ROOT = '/content/54_20-master'
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Verify P0 gate
import json
from pathlib import Path

p0_gate_path = Path('handoff/P0_GATE.json')
if p0_gate_path.exists():
    with open(p0_gate_path) as f:
        p0_gate = json.load(f)
    print(f'P0 Gate Status: {p0_gate["training_allowed"]}')
    if not p0_gate['training_allowed']:
        raise RuntimeError('P0 gate not passed! Training not allowed.')
else:
    print('P0 gate file not found, proceeding with caution')

In [ ]:
# Configuration
TRAINING_SEEDS = [1101, 2202, 3303]
VARIANTS = ['GPPO-NoGate', 'GPPO-Adaptive', 'PPO-MLP']
TIMESTEPS = 300_000  # Per model per seed
CHECKPOINT_INTERVAL = 25_000
EVENTS_PER_EPISODE = 5

print(f'Training seeds: {TRAINING_SEEDS}')
print(f'Variants: {VARIANTS}')
print(f'Timesteps per model per seed: {TIMESTEPS}')
print(f'Total training runs: {len(TRAINING_SEEDS) * len(VARIANTS)}')

In [ ]:
# Import project modules
import sys
sys.path.insert(0, 'ppo_allocation')

from random_event.experiment import (
    CyclingTrainingEnv,
    train_variants,
    generate_protocol_bank,
    evaluate_tape_bank,
)
from random_event.trainer import PPOConfig
from random_event.models import make_no_gate_model, make_adaptive_model, make_fair_ppo_mlp

print('Project modules imported successfully')

In [ ]:
# Generate validation bank (for checkpoint selection)
print('Generating validation bank...')
output_dir = Path('results/random_event')
output_dir.mkdir(parents=True, exist_ok=True)

validation_manifest = generate_protocol_bank(
    output_dir,
    tier='preliminary',
    split='validation',
    events_per_tape=5,
    limit_per_set=4,  # Small for testing
)
print(f'Validation bank: {validation_manifest["tape_count"]} tapes')

In [ ]:
# Generate test bank (for final evaluation only)
print('Generating test bank...')
test_manifest = generate_protocol_bank(
    output_dir,
    tier='preliminary',
    split='test',
    events_per_tape=5,
    limit_per_set=4,  # Small for testing
)
print(f'Test bank: {test_manifest["tape_count"]} tapes')

In [ ]:
# Train all variants
print('Starting training...')
training_results = train_variants(
    output_dir,
    variants=VARIANTS,
    seeds=TRAINING_SEEDS,
    timesteps=TIMESTEPS,
    events_per_episode=EVENTS_PER_EPISODE,
)
print(f'Training completed: {len(training_results["records"])} checkpoints saved')

In [ ]:
# Evaluate on validation bank
print('Evaluating on validation bank...')
validation_results = evaluate_tape_bank(
    output_dir / 'validation_eval',
    manifest_path=validation_manifest['manifest_path'],
    gppo_checkpoints=[output_dir / 'models'],
)
print(f'Validation evaluation completed')

In [ ]:
# Select best checkpoint based on validation
# (In practice, this would be done programmatically)
print('Checkpoint selection based on validation results...')
# For now, we'll use the last checkpoint from each variant
selected_checkpoints = {}
for variant in VARIANTS:
    for seed in TRAINING_SEEDS:
        safe_variant = variant.lower().replace('-', '_')
        checkpoint = output_dir / 'models' / f'{safe_variant}_seed{seed}_steps{TIMESTEPS}.pt'
        if checkpoint.exists():
            selected_checkpoints[(variant, seed)] = checkpoint
            print(f'  Selected: {checkpoint.name}')

In [ ]:
# Final evaluation on test bank
print('Final evaluation on test bank...')
test_results = evaluate_tape_bank(
    output_dir / 'test_eval',
    manifest_path=test_manifest['manifest_path'],
    gppo_checkpoints=[selected_checkpoints[(v, s)] for v, s in selected_checkpoints],
)
print(f'Test evaluation completed')

In [ ]:
# Generate summary statistics
print('\n=== Preliminary Results Summary ===')
print(f'Label: preliminary')
print(f'Training seeds: {TRAINING_SEEDS}')
print(f'Test tapes: {test_manifest["tape_count"]}')
print()

# Print key metrics
for algorithm, records in test_results['summaries'].items():
    metrics = records['metrics']
    return_mean = metrics.get('episode_return', {}).get('mean', 0)
    success_rate = metrics.get('event_success_rate', {}).get('mean', 0)
    print(f'{algorithm}:')
    print(f'  Episode Return: {return_mean:.3f}')
    print(f'  Success Rate: {success_rate:.3f}')

In [ ]:
# Save results to Drive
import shutil

results_to_save = [
    ('results/random_event/training_summary.json', 'training_summary.json'),
    ('results/random_event/test_eval/evaluation_summary.json', 'test_evaluation_summary.json'),
    ('handoff/P0_GATE.json', 'P0_gate.json'),
]

for source, dest in results_to_save:
    if Path(source).exists():
        shutil.copy(source, os.path.join(DRIVE_PATH, dest))
        print(f'Saved: {dest}')

print(f'\nAll results saved to: {DRIVE_PATH}')

In [ ]:
# Print completion message
print('\n' + '='*60)
print('PRELIMINARY TRAINING COMPLETED')
print('='*60)
print(f'Label: preliminary')
print(f'Training seeds: {TRAINING_SEEDS}')
print(f'Variants: {VARIANTS}')
print(f'Timesteps: {TIMESTEPS}')
print()
print('IMPORTANT: This is a preliminary run with only 3 seeds.')
print('Results should not be used as final conclusions.')
print('A formal run with 5 seeds is required for publication.')